# Eurovision-like Jury Results

This notebook computes the final jury results from a long-format vote table.
Each step is in its own section so intermediate tables are easy to inspect.

## Step 1 — Load and pivot the juror votes

**Input:** `jury_votes.xlsx` in long format with columns `Voting_country`, `Juror`, `Participating_country`, `Rank`.

**Output (`ranks_df`):** a wide DataFrame where:
- rows are **participating countries** (in source order),
- columns are a 2-level `MultiIndex` `(Voting_country, Juror)`, grouped by voting country.

Each cell holds the rank that juror gave to that participating country. A `0` means the juror is from that country (self-vote).

In [1]:
import pandas as pd

INPUT_FILE = 'jury_votes.xlsx'

long_df = pd.read_excel(INPUT_FILE)
long_df.head()

,Voting_country,Juror,Participating_country,Rank
0,FR,Juror 1,GB,3
1,FR,Juror 2,GB,3
2,FR,Juror 3,GB,3
3,FR,Juror 1,ES,1
4,FR,Juror 2,ES,1


In [2]:
# Capture source order so the pivot keeps it (pandas would otherwise sort alphabetically)
participating_order = long_df['Participating_country'].drop_duplicates().tolist()
voting_order        = long_df['Voting_country'].drop_duplicates().tolist()
juror_order         = long_df['Juror'].drop_duplicates().tolist()

ranks_df = (
    long_df
    .pivot(index='Participating_country',
           columns=['Voting_country', 'Juror'],
           values='Rank')
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=['Voting_country', 'Juror']))
)

ranks_df

Voting_country             FR                      GB                      DE  \
Juror                 Juror 1 Juror 2 Juror 3 Juror 1 Juror 2 Juror 3 Juror 1   
Participating_country                                                           
GB                          3       3       3       0       0       0       1   
ES                          1       1       1       1       3       2       3   
FR                          0       0       0       2       2       3       2   
DE                          2       2       2       3       1       1       0   

Voting_country                             ES                      PT          \
Juror                 Juror 2 Juror 3 Juror 1 Juror 2 Juror 3 Juror 1 Juror 2   
Participating_country                                                           
GB                          1       1       2       2       1       3       1   
ES                          2       3       0       0       0       4       4   
FR                          3       2       1       3       3       2       2   
DE                          0       0       3       1       2       1       3   

Voting_country                 
Juror                 Juror 3  
Participating_country          
GB                          3  
ES                          2  
FR                          4  
DE                          1

In [3]:
# Quick sanity checks
print('Shape :', ranks_df.shape)
print('NaNs  :', ranks_df.isna().sum().sum(), '(should be 0)')
print('Self-votes (rank 0) per voting country:')
(ranks_df == 0).sum().groupby(level='Voting_country').sum()

Shape : (4, 15)
NaNs  : 0 (should be 0)
Self-votes (rank 0) per voting country:


Voting_country
DE    3
ES    3
FR    3
GB    3
PT    0
dtype: int64